### Autoencoder Implementation with Keras

An autoencoder is a type of artificial neural network used for learning efficient codings of unlabeled data (unsupervised learning). The aim of an autoencoder is to learn a representation (encoding) for a set of data, typically for dimensionality reduction, by training the network to ignore signal 'noise'.

Here, we'll implement a simple autoencoder using Keras.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

#### 1. Define the Autoencoder Model

We'll create a simple autoencoder with an encoder and a decoder. For demonstration, let's assume our input data will be 28x28 grayscale images (like MNIST digits), which we'll flatten to 784 dimensions.

In [2]:
# Define input shape (e.g., for flattened 28x28 images)
input_dim = 784 # 28 * 28
encoding_dim = 32 # Dimension of the latent space

# Encoder part
encoder_input = keras.Input(shape=(input_dim,))
encoded = layers.Dense(128, activation='relu')(encoder_input)
encoded = layers.Dense(encoding_dim, activation='relu')(encoded)

# Decoder part
decoder_output = layers.Dense(128, activation='relu')(encoded)
decoder_output = layers.Dense(input_dim, activation='sigmoid')(decoder_output)

# Autoencoder model
autoencoder = keras.Model(encoder_input, decoder_output)

# Encoder model (for extracting the latent representation)
encoder = keras.Model(encoder_input, encoded)

# Decoder model (for reconstructing from the latent representation)
encoded_input = keras.Input(shape=(encoding_dim,))
decoder_layer = autoencoder.layers[-2](encoded_input) # Reuse the second-to-last layer of the autoencoder
decoder_layer = autoencoder.layers[-1](decoder_layer) # Reuse the last layer
decoder = keras.Model(encoded_input, decoder_layer)

autoencoder.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 784)            │       101,136 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 209,968 (820.19 KB)

 Trainable params: 209,968 (820.19 KB)

 Non-trainable params: 0 (0.00 B)

#### 2. Compile the Autoencoder

We'll use a `binary_crossentropy` loss function (common for image reconstruction where pixel values are normalized between 0 and 1) and the Adam optimizer.

In [3]:
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

#### 3. Prepare Dummy Data for Training

For demonstration, we'll create some random dummy data that resembles the shape of our expected input (e.g., flattened images). In a real scenario, you would load and preprocess your actual dataset.

In [4]:
# Generate some dummy data (e.g., 1000 samples of 784 dimensions, normalized)
x_train = np.random.rand(1000, input_dim).astype('float32')

print(f"Shape of dummy training data: {x_train.shape}")

Shape of dummy training data: (1000, 784)


#### 4. Train the Autoencoder

An autoencoder is trained to reconstruct its input. So, `x_train` is used as both the input and the target (`y_train`).

In [5]:
history = autoencoder.fit(x_train, x_train, epochs=10, batch_size=256, shuffle=True, validation_split=0.2)

print("Autoencoder training complete.")

Epoch 1/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 81ms/step - loss: 0.6937 - val_loss: 0.6932
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 0.6931 - val_loss: 0.6932
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.6931 - val_loss: 0.6932
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.6931 - val_loss: 0.6932
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 0.6930 - val_loss: 0.6932
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - loss: 0.6930 - val_loss: 0.6932
Epoch 7/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - loss: 0.6930 - val_loss: 0.6933
Epoch 8/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 0.6929 - val_loss: 0.6933
Epoch 9/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 0.6928 - val_loss: 0.6932
Epoch 10/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 0.6928 - val_loss: 0.6934
Autoencoder training complete.
